# 03 - Extract HPOA Attributes and Resolve Human-Readable Labels

## Learning objectives
- Extract `Disease`, `Phenotypic feature`, `Author`, `Date`, and `Source` from `phenotype_local.hpoa`.
- Build the target tuple example for `OMIM:222100` and `HP:0410050`.
- Load `hp.owl` with RDFLib and resolve human-readable labels for HPO terms.
- Use the website fallback only for `HP:0410050` and `HP:0100651` if ontology matching fails.

In [1]:
from pathlib import Path
import re
from urllib.request import urlopen

import pandas as pd
from rdflib import Graph, URIRef
from rdflib.namespace import RDFS

In [2]:
# Resolve project root so the notebook works from both project root and notebooks/.
cwd = Path.cwd()
if (cwd / "data" / "phenotype_local.hpoa").exists():
    project_root = cwd
elif (cwd.parent / "data" / "phenotype_local.hpoa").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Could not find data/phenotype_local.hpoa from current working directory.")

hpoa_path = project_root / "data" / "phenotype_local.hpoa"
owl_path = project_root / "data" / "hp.owl"

df_hpoa = pd.read_csv(hpoa_path, sep='\t', dtype='string')
print(f"Loaded HPOA rows: {len(df_hpoa):,}")

Loaded HPOA rows: 280,730


In [4]:
def camel_case_to_words(text: str) -> str:
    """Convert strings like 'NicoleVasilevsky' to 'Nicole Vasilevsky'."""
    if not text:
        return text
    text = text.replace('_', ' ')
    return re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text).strip()


def parse_first_biocuration(value: str):
    """
    Parse first curator/date from values like:
    HPO:NicoleVasilevsky[2018-02-23];HPO:NicoleVasilevsky[2018-03-02]
    """
    if pd.isna(value) or not str(value).strip():
        return pd.NA, pd.NA

    first_entry = str(value).split(';')[0].strip()
    match = re.match(r'^HPO:([^\[]+)\[([0-9]{4}-[0-9]{2}-[0-9]{2})\]$', first_entry)
    if not match:
        return pd.NA, pd.NA

    author_raw, date_raw = match.groups()
    return camel_case_to_words(author_raw), date_raw


def first_source(value: str):
    """Pick the first source from semicolon-separated references."""
    if pd.isna(value) or not str(value).strip():
        return pd.NA
    return str(value).split(';')[0].strip()

In [5]:
# Extract the requested attributes into a compact DataFrame.
df_extracted = df_hpoa[["database_id", "hpo_id", "reference", "biocuration"]].copy()
df_extracted[["Author", "Date"]] = df_extracted["biocuration"].apply(
    lambda x: pd.Series(parse_first_biocuration(x))
)
df_extracted["Source"] = df_extracted["reference"].apply(first_source)

df_extracted = df_extracted.rename(columns={
    "database_id": "Disease",
    "hpo_id": "Phenotypic feature",
})[["Disease", "Phenotypic feature", "Author", "Date", "Source"]]

df_extracted.head()

,Disease,Phenotypic feature,Author,Date,Source
0,OMIM:619340,HP:0011097,probinson,2021-06-21,PMID:31675180
1,OMIM:619340,HP:0002187,probinson,2021-06-21,PMID:31675180
2,OMIM:619340,HP:0001518,probinson,2021-06-21,PMID:31675180
3,OMIM:619340,HP:0032792,probinson,2021-06-21,PMID:31675180
4,OMIM:619340,HP:0011451,probinson,2021-06-21,PMID:31675180


In [6]:
# Build the exact example tuple requested in the prompt.
target_disease = "OMIM:222100"
target_feature = "HP:0410050"
check_term = "HP:0100651"

target_row = df_extracted[(df_extracted["Disease"] == target_disease) &
                          (df_extracted["Phenotypic feature"] == target_feature)].iloc[0]

example_tuple = (
    f"{target_row['Disease']} (check {check_term}) "
    f"{target_row['Phenotypic feature']} {target_row['Author']} "
    f"{target_row['Date']} {target_row['Source']}"
)

print(example_tuple)
# Expected shape: OMIM:222100 (check HP:0100651) HP:0410050 Nicole Vasilevsky 2018-02-23 PMID:9357814

OMIM:222100 (check HP:0100651) HP:0410050 Nicole Vasilevsky 2018-02-23 PMID:9357814


In [7]:
# Load ontology.
g = Graph()
g.parse(owl_path)
print(f"Loaded ontology triples: {len(g):,}")

Loaded ontology triples: 905,482


In [8]:
def term_uri(term_id: str) -> URIRef:
    """Convert HP:0410050 -> http://purl.obolibrary.org/obo/HP_0410050"""
    return URIRef(f"http://purl.obolibrary.org/obo/{term_id.replace(':', '_')}")


def label_from_ontology(graph: Graph, term_id: str):
    """Resolve rdfs:label directly from hp.owl for an HPO ID."""
    label = graph.value(term_uri(term_id), RDFS.label)
    return str(label) if label else None


def label_from_hpo_website(term_id: str):
    """
    Website fallback constrained to the two IDs requested by the prompt.
    URLs checked only when ontology lookup fails.
    """
    allowed = {"HP:0410050", "HP:0100651"}
    if term_id not in allowed:
        return None

    url = f"https://hpo.jax.org/browse/term/{term_id}"
    try:
        with urlopen(url, timeout=15) as response:
            html = response.read().decode("utf-8", errors="ignore")
    except Exception:
        return None

    # Try a few robust patterns, then clean website suffixes.
    patterns = [
        r'<meta\s+property="og:title"\s+content="([^"]+)"',
        r'<h1[^>]*>([^<]+)</h1>',
        r'<title>([^<]+)</title>',
    ]

    for pattern in patterns:
        match = re.search(pattern, html, flags=re.IGNORECASE)
        if not match:
            continue

        candidate = match.group(1).strip()
        candidate = re.sub(r'\s*[|\-]\s*Human Phenotype Ontology.*$', '', candidate).strip()
        candidate = candidate.replace(term_id, '').strip(' -|')
        if candidate:
            return candidate

    return None


def resolve_hpo_label(graph: Graph, term_id: str):
    """Resolve label from ontology first, then website fallback for allowed terms."""
    label = label_from_ontology(graph, term_id)
    if label:
        return label
    return label_from_hpo_website(term_id)

In [9]:
# Build a new tuple with human-readable labels.
# Disease is OMIM-based in this row, so we use the check term (HP:0100651) to get an HPO label for disease context.
disease_label = resolve_hpo_label(g, check_term)
feature_label = resolve_hpo_label(g, target_feature)

readable_tuple = (
    target_row['Disease'],
    disease_label if disease_label else f"[No HP label found for {check_term}]",
    target_row['Phenotypic feature'],
    feature_label if feature_label else f"[No HP label found for {target_feature}]",
    target_row['Author'],
    target_row['Date'],
    target_row['Source'],
)

print("Tuple with labels:")
print(readable_tuple)

Tuple with labels:
('OMIM:222100', 'Type I diabetes mellitus', 'HP:0410050', 'Decreased level of 1,5 anhydroglucitol in serum', 'Nicole Vasilevsky', '2018-02-23', 'PMID:9357814')


## Notes
- For this specific example, both `HP:0100651` and `HP:0410050` are available in `hp.owl`, so fallback is normally not needed.
- The fallback is intentionally restricted to:
  - `https://hpo.jax.org/browse/term/HP:0410050`
  - `https://hpo.jax.org/browse/term/HP:0100651`

## Exercises
1. For `OMIM:222100`, build a tuple for `HP:0000819` (same tuple format used in the example).
2. Count how many rows in `df_extracted` have `Author == "Nicole Vasilevsky"`.
3. Create a small DataFrame with labels for these terms: `HP:0410050`, `HP:0100651`, and `HP:0001250`.